In [1]:
# Create LSTM model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import logging
import time
from dataloader import FloodDataGenerator
import matplotlib.pyplot as plt
import json


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("LSTM_model_training")

In [2]:
# Define Constants and V

LAG = 8 # 8time steps of history. 2 hours  
FORECAST_HORIZON = 1 # Predict 1 time steps ahead. Next 15 minutes
TRAINING_BATCH_SIZE = 10000 # Number of samples per batch used for training

TRAIN_SUBSET_IDENTIFIER = 'train'
VAL_SUBSET_IDENTIFIER = 'validation'


In [4]:
def create_lstm_model(num_features):
    input_shape = (LAG, num_features)  # (timesteps, features)
    model = Sequential([
        LSTM(64, input_shape=input_shape, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(FORECAST_HORIZON)  # Output shape matches horizon
    ])
    
    # Compile model
    model.compile(optimizer='adam', loss='mse', metrics=['mse'])
    return model
    
def create_data_generators(lag, horizon, batch_size):
    train_generator = FloodDataGenerator(batch_size,lag, horizon, TRAIN_SUBSET_IDENTIFIER)
    val_generator = FloodDataGenerator(batch_size, lag, horizon, VAL_SUBSET_IDENTIFIER)
    return train_generator, val_generator

In [5]:
# Train the model
def train_lstm_model(model, train_generator, val_generator):
    logger.info("Training LSTM model")
    model = create_lstm_model(num_features)
    start = time.time()
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=2,
        verbose=1
    )
    end = time.time()
    logger.info(f"Training completed in {end-start} seconds")
    return model, history

def plot_model_history(history):
    # Plot the model history and save it as a figure and json file
    if not history:
        logger.error("Model history not found")
        return
    
    logger.info("Plotting model history")
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('Model Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='upper left')
    plt.savefig('model_history.png')

    history_dict = history.history
    with open('model_history.json', 'w') as f:
        json.dump(history_dict, f)

if __name__ == "__main__":
    logger.info("Model trainer started")
    train_generator, val_generator, num_features = create_data_generators(LAG, FORECAST_HORIZON, TRAINING_BATCH_SIZE)
    model = create_lstm_model(num_features)
    model, history = train_lstm_model(model, train_generator, val_generator)
    plot_model_history(history)
    model.save("lstm_model.h5")
    logger.info("Model saved as lstm_model.h5")

INFO:LSTM_model_training:Model trainer started


OperationalError: could not connect to server: Connection refused
	Is the server running on host "localhost" (::1) and accepting
	TCP/IP connections on port 5432?
could not connect to server: Connection refused
	Is the server running on host "localhost" (127.0.0.1) and accepting
	TCP/IP connections on port 5432?
